# Faster R-CNN / Mask R-CNN — Colab Training

In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU found — change runtime to T4'
print('GPU:', torch.cuda.get_device_name(0))
print('torch version:', torch.__version__)

In [ ]:
!git clone https://github.com/martytcoleman/PA3.git
%cd PA3

In [ ]:
!pip install tqdm pyyaml opencv-python-headless --quiet

In [ ]:
import os

!wget -q --show-progress http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
!wget -q --show-progress http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
!tar -xf VOCtrainval_06-Nov-2007.tar
!tar -xf VOCtest_06-Nov-2007.tar

for split, txt in [('VOC2007', 'trainval'), ('VOC2007-test', 'test')]:
    os.makedirs(f'{split}/JPEGImages', exist_ok=True)
    os.makedirs(f'{split}/Annotations', exist_ok=True)
    with open(f'VOCdevkit/VOC2007/ImageSets/Main/{txt}.txt') as f:
        for line in f:
            img_id = line.strip()
            for ext, sub in [('.jpg', 'JPEGImages'), ('.xml', 'Annotations')]:
                src = os.path.abspath(f'VOCdevkit/VOC2007/{sub}/{img_id}{ext}')
                dst = f'{split}/{sub}/{img_id}{ext}'
                if not os.path.exists(dst):
                    os.symlink(src, dst)

print('trainval images:', len(os.listdir('VOC2007/JPEGImages')))
print('test images:', len(os.listdir('VOC2007-test/JPEGImages')))

In [ ]:
import sys
sys.path.insert(0, '.')
from train.test_implementation import (
    test_iou, test_sample_positive_negative,
    test_clamp_boxes, test_transform_boxes,
    test_rpn, test_roi_head, test_full_model
)
test_iou()
test_sample_positive_negative()
test_clamp_boxes()
test_transform_boxes()
test_rpn()
test_roi_head()
test_full_model()
print('all tests passed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/PA3_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('drive mounted, saving to:', DRIVE_DIR)

In [ ]:
!python train/train_faster_rcnn.py --config config/voc.yaml

In [ ]:
import shutil
shutil.copy('voc/faster_rcnn_voc2007.pth', f'{DRIVE_DIR}/faster_rcnn_voc2007.pth')
print('faster r-cnn checkpoint saved to drive')

In [ ]:
!python test/test_faster_rcnn.py --config config/voc.yaml --evaluate True --infer_samples True

In [ ]:
!python train/train_mask_rcnn.py --config config/voc_mask.yaml

In [ ]:
shutil.copy('voc_mask/mask_rcnn_voc2007.pth', f'{DRIVE_DIR}/mask_rcnn_voc2007.pth')
print('mask r-cnn checkpoint saved to drive')